In [42]:
import src.digest_util as dig
import src.dna_util as dna
import src.files_util as files


#### Load fasta files for further use

In [43]:
genomes = {'e.coli': [i for i in files.load_fasta_file('fasta/E.Coli/GCF_000005845.2_ASM584v2_genomic.fna')],
           'yeast': [i for i in files.load_fasta_file('fasta/Yeast/GCF_000146045.2_R64_genomic.fna')],
           #'roundworm': [i for i in files.load_fasta_file('fasta/Roundworm/GCF_000002985.6_WBcel235_genomic.fna')]
           }

#### List sequences for each organism

In [44]:
# labels
labels = {}
for key, value in genomes.items():
    labels[key] = [i[0] for i in value]

print(labels) # intentionaly using printing for constency

{'e.coli': ['NC_000913.3 Escherichia coli str. K-12 substr. MG1655, complete genome'], 'yeast': ['NC_001133.9 Saccharomyces cerevisiae S288C chromosome I, complete sequence', 'NC_001134.8 Saccharomyces cerevisiae S288C chromosome II, complete sequence', 'NC_001135.5 Saccharomyces cerevisiae S288C chromosome III, complete sequence', 'NC_001136.10 Saccharomyces cerevisiae S288C chromosome IV, complete sequence', 'NC_001137.3 Saccharomyces cerevisiae S288C chromosome V, complete sequence', 'NC_001138.5 Saccharomyces cerevisiae S288C chromosome VI, complete sequence', 'NC_001139.9 Saccharomyces cerevisiae S288C chromosome VII, complete sequence', 'NC_001140.6 Saccharomyces cerevisiae S288C chromosome VIII, complete sequence', 'NC_001141.2 Saccharomyces cerevisiae S288C chromosome IX, complete sequence', 'NC_001142.9 Saccharomyces cerevisiae S288C chromosome X, complete sequence', 'NC_001143.9 Saccharomyces cerevisiae S288C chromosome XI, complete sequence', 'NC_001144.5 Saccharomyces cerev

In [45]:
rand_seq = []

# length in Mega Bases
for length in [4, 8]:#, 14, 28]:
    # using weird seed just for repeatable results and different sequences
    # without different seeds first 4 Mb will be the same in each sequence
    rand_seq.append(tuple(['rand_gen_len_'+str(length),
                          dna.random_genome(lenght=length*1000000,
                                            seed=length*15)]))
for i in rand_seq:
    print(i[0]+' : '+i[1][:400])

rand_gen_len_4 : CCGGTTACCGCTGCAACGAGTAGCAAGTTGAATAGGCGTCGGCCTATGGCATTCATATAGTGGATGAACGAGCTGGTGGTTCTCTTTGGAGTAGATCTTAGTTTGCGAGCTGAATGTCACGAAACGTTACGGCCATCAGCAGAGGTTCACTATAAAACACCCAGGAGTGAAGGGGTTCGGCTGGGTCAAGTAGGACTTTCGATCACTCAAGCTGTGTTCGAGAAGAGGCTGTGGAGGGTGGGGGTCGCGCTATGGGCCGTCCTTCTATGTTGAAGAACCGGCCACGAGGACTGTACCCTTCAGAGAGATACGGGGATTGCGTCTATCAACACTTAACCGTGTATTAGTAGCTGGGAACGTCTCTCTTAACGACAACCCAGCCGCCGATCCACTGTCCGTC
rand_gen_len_8 : TAGTGGTAACATTCTACACCGGTCCCCACTCGGGAAAGTTCAGGCGAACGACCTGACTCTCAAGCGCTCGGCCTATAATGCTACGACAGGACAGCGCGGATGTTTACTTCCGCGGGCCCAAATCCGCGCTGAAATAGATGGCTGATTAAGCGTAGGCGTCACTCACGTTTGTCGGAGAACGTTTTAAGCATACAATGCTTTGCCTTCGGTCAAGTGGTCTTGGATCGGGAACCGTCGGGTGGTACGAGCAGTTACTAGATCAGTCCAGCGGTGAGACGAACAAGTGGTCGATTCGCTGTCAAACGGGATGCTAATAGAGATATTGTTATAGTAAAAGGATCGGCTGTGCCGTCTACACAATCTAAGTAGGTGTTATAAGAAGACCTAGAAGATAGATTAT


In [46]:
# add random genomes to genomes dict
genomes['random'] = rand_seq

# digest all sequences - BspQI (7), BssSI (6)
digest = {}

def dig_with_enzyme(sequences, enzyme):
    tmp = {}
    for species, genome in sequences.items():
        rmaps = []
        for chromosome in genome:
            rmaps.append([chromosome[0], dig.digest_with_enzyme(chromosome[1], enzyme)])
            
        tmp[species] = rmaps

    return tmp

digest['BspQI (GCTCTTC)'] = dig_with_enzyme(genomes, 'BspQI')
digest['BssSI (CACGAG)'] = dig_with_enzyme(genomes, 'BssSI')


def print_dict(digest_dict):
    for enzyme, dig in digest_dict.items():
        print('enzyme : '+enzyme)
        for species, genome in dig.items():
            print('\tspecies : '+species)
            for rmap in genome:
                if isinstance(rmap[1], str):
                    print('\t\t'+rmap[0]+' :\n'+rmap[1][0:200])
                else:
                    print('\t\t'+rmap[0]+' :\n'+rmap[1])

print_dict(digest)
    

enzyme : BspQI (GCTCTTC)
	species : e.coli
		NC_000913.3 Escherichia coli str. K-12 substr. MG1655, complete genome :
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
	species : yeast
		NC_001133.9 Saccharomyces cerevisiae S288C chromosome I, complete sequence :
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
		NC_001134.8 Saccharomyces cerevisiae S288C chromosome II, complete sequence :
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
		NC_001135.5 Saccharomyces cerevisiae S288C chromosome III, complete sequence :
0000000000000000000000

In [49]:
def quant_with_value(maps, quant):
    tmp = {}
    for enzyme, sequences in maps.items():
        quanted = {}
        for species, genome in sequences.items():
            rmaps = []
            for chromosome in genome:
                rmaps.append([chromosome[0], dig.gen_bin_map_from_digest(chromosome[1], quantize=quant)])
            quanted[species] = rmaps
        tmp[enzyme] = quanted

    return tmp

quanted_maps = quant_with_value(digest, quant=250)

print_dict(quanted_maps)

enzyme : BspQI (GCTCTTC)
	species : e.coli
		NC_000913.3 Escherichia coli str. K-12 substr. MG1655, complete genome :
00000000000000000000000000000000000000000001000000000000000000000000000000000000000000000000000000000000000000000000000000000000001000000000000000000000000000001000000000000000000000000000000000000000
	species : yeast
		NC_001133.9 Saccharomyces cerevisiae S288C chromosome I, complete sequence :
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000
		NC_001134.8 Saccharomyces cerevisiae S288C chromosome II, complete sequence :
00000000000000000000000000000000000000000001000000000010000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000010000000000000000000000000000000000000001000000000000
		NC_001135.5 Saccharomyces cerevisiae S288C chromosome III, complete sequence :
0000000010000000000100

In [48]:
def frequency(sequence):
    count = 0
    for i in sequence:
        if i == '1':
            count = count + 1
    return count/len(sequence)

def count_labels_freq(maps):
    tmp = {}
    for enzyme, sequences in maps.items():
        quanted = {}
        for species, genome in sequences.items():
            rmaps = []
            for chromosome in genome:
                rmaps.append([chromosome[0], frequency(chromosome[1])])
            quanted[species] = rmaps
        tmp[enzyme] = quanted

    return tmp


freq_of_maps = count_labels_freq(quanted_maps)

print_dict(freq_of_maps)

enzyme : BspQI (GCTCTTC)
	species : e.coli
		NC_000913.3 Escherichia coli str. K-12 substr. MG1655, complete genome :
1's freq = 0.012215352133916453
	species : yeast
		NC_001133.9 Saccharomyces cerevisiae S288C chromosome I, complete sequence :
1's freq = 0.010855405992184108
		NC_001134.8 Saccharomyces cerevisiae S288C chromosome II, complete sequence :
1's freq = 0.011436301032956222
		NC_001135.5 Saccharomyces cerevisiae S288C chromosome III, complete sequence :
1's freq = 0.01484054310072624
		NC_001136.10 Saccharomyces cerevisiae S288C chromosome IV, complete sequence :
1's freq = 0.01318537859007833
		NC_001137.3 Saccharomyces cerevisiae S288C chromosome V, complete sequence :
1's freq = 0.014733922690240943
		NC_001138.5 Saccharomyces cerevisiae S288C chromosome VI, complete sequence :
1's freq = 0.014803849000740192
		NC_001139.9 Saccharomyces cerevisiae S288C chromosome VII, complete sequence :
1's freq = 0.011365719523373053
		NC_001140.6 Saccharomyces cerevisiae S288C chrom